# CS-781 Phase 4 - ensemble of the AllRuns outputs
Input: output of the *Phase4 cjpe allruns* notebook. CPU is enough. Writes `submissions/ensemble_greedy.zip` and `submissions/ensemble_top5.zip`.

In [1]:
# Phase 4 ensemble (CPU is enough). Input: the output of the "Phase4 cjpe allruns" notebook.
import os, json, zipfile
import numpy as np

def macro_f1(y, pred):
    fs = []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        fs.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(fs))

def best_threshold(y, p):
    cands = np.unique(np.concatenate([np.quantile(p, np.linspace(0.02, 0.98, 97)), [0.5]]))
    best = (macro_f1(y, (p >= 0.5).astype(int)), 0.5)
    for t in cands:
        f = macro_f1(y, (p >= t).astype(int))
        if f > best[0] + 1e-9:
            best = (f, float(t))
    return best[1], best[0]

def honest(y, p, seed=0):
    idx = np.random.RandomState(seed).permutation(len(y)); h = len(idx) // 2; b, a = [], []
    for tr, te in ((idx[:h], idx[h:]), (idx[h:], idx[:h])):
        t, _ = best_threshold(y[tr], p[tr])
        b.append(macro_f1(y[te], (p[te] >= .5).astype(int))); a.append(macro_f1(y[te], (p[te] >= t).astype(int)))
    return float(np.mean(b)), float(np.mean(a))

# only the Phase-4 run folders (cj_<method>_<arch>, no smoke runs)
runs = {}
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    name = os.path.basename(root)
    if name.startswith("cj_") and not name.endswith("_smoke") and {"probs_dev.json", "probs_test.json"} <= set(files):
        runs.setdefault(name, (json.load(open(os.path.join(root, "probs_dev.json")))["docs"],
                               json.load(open(os.path.join(root, "probs_test.json")))["docs"]))
print("runs found:", sorted(runs))
names = sorted(runs)
dev_ids = sorted(set.intersection(*[set(runs[n][0]) for n in names]))
test_ids = sorted(set.intersection(*[set(runs[n][1]) for n in names]))
print("dev docs", len(dev_ids), "| test docs", len(test_ids))
y = np.array([runs[names[0]][0][i][1] for i in dev_ids])
PD = {n: np.array([runs[n][0][i][0] for i in dev_ids]) for n in names}
PT = {n: np.array([runs[n][1][i][0] for i in test_ids]) for n in names}
for n in names:
    print(f"{n:>22}  F1@0.5 {macro_f1(y, (PD[n] >= .5).astype(int)):.4f}  tuned {best_threshold(y, PD[n])[1]:.4f}  honest {honest(y, PD[n])[1]:.4f}")

members, best = [], -1.0
for _ in range(10):
    f, n = max((best_threshold(y, np.mean([PD[m] for m in members + [n]], 0))[1], n) for n in names)
    if f <= best + 1e-6:
        break
    members.append(n); best = f
    print(f"+ {n:>22} -> dev F1 (tuned) {f:.4f}")
pd_, pt_ = np.mean([PD[m] for m in members], 0), np.mean([PT[m] for m in members], 0)
thr, f = best_threshold(y, pd_)
h0, h1 = honest(y, pd_)
thr = thr if h1 >= h0 else 0.5
print(f"ENSEMBLE {members}: F1@0.5 {macro_f1(y, (pd_ >= .5).astype(int)):.4f} | tuned {f:.4f} (thr {thr:.3f}) | honest {h0:.4f}->{h1:.4f}")

# also the plain average of the five strongest single runs (a less dev-fitted alternative)
top5 = sorted(names, key=lambda n: -honest(y, PD[n])[1])[:5]
pd5, pt5 = np.mean([PD[m] for m in top5], 0), np.mean([PT[m] for m in top5], 0)
thr5, f5 = best_threshold(y, pd5); g0, g1 = honest(y, pd5); thr5 = thr5 if g1 >= g0 else 0.5
print(f"TOP5 AVG {top5}: tuned {f5:.4f} (thr {thr5:.3f}) | honest {g0:.4f}->{g1:.4f}")

os.makedirs("/kaggle/working/submissions", exist_ok=True)
for tag, p, t in (("ensemble_greedy", pt_, thr), ("ensemble_top5", pt5, thr5)):
    out = [{"id": i, "pred": int(v >= t)} for i, v in zip(test_ids, p)]
    json.dump(out, open("/kaggle/working/prediction.json", "w"))
    with zipfile.ZipFile(f"/kaggle/working/submissions/{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("/kaggle/working/prediction.json", arcname="prediction.json")
    print(tag, "| test 1-rate", round(float(np.mean([o["pred"] for o in out])), 3))
os.remove("/kaggle/working/prediction.json")
print(sorted(os.listdir("/kaggle/working/submissions")))


runs found: ['cj_adalora_retnet', 'cj_adapter_encdec', 'cj_bitfit_mla', 'cj_full_gqa', 'cj_kd_gqa', 'cj_lorma_gqa', 'cj_partial_retnet']
dev docs 2511 | test docs 1517
     cj_adalora_retnet  F1@0.5 0.6263  tuned 0.6458  honest 0.6390
     cj_adapter_encdec  F1@0.5 0.3350  tuned 0.5798  honest 0.5732
         cj_bitfit_mla  F1@0.5 0.6142  tuned 0.6507  honest 0.6449
           cj_full_gqa  F1@0.5 0.6562  tuned 0.6905  honest 0.6838
             cj_kd_gqa  F1@0.5 0.6117  tuned 0.6618  honest 0.6540
          cj_lorma_gqa  F1@0.5 0.6491  tuned 0.6865  honest 0.6854
     cj_partial_retnet  F1@0.5 0.6177  tuned 0.6694  honest 0.6609
+            cj_full_gqa -> dev F1 (tuned) 0.6905
+           cj_lorma_gqa -> dev F1 (tuned) 0.7023
ENSEMBLE ['cj_full_gqa', 'cj_lorma_gqa']: F1@0.5 0.6599 | tuned 0.7023 (thr 0.335) | honest 0.6599->0.7021
TOP5 AVG ['cj_lorma_gqa', 'cj_full_gqa', 'cj_partial_retnet', 'cj_kd_gqa', 'cj_bitfit_mla']: tuned 0.6890 (thr 0.360) | honest 0.6380->0.6743
ensemble_greed